# Lesson 6: Ask, don't guess

**Project:** Agentic Engineering Worklog

**Model:** `qwen3:8b`, running on this PC through Ollama

**Part 2, Make it trustworthy.** Lessons 1 to 5 built an agent and its harness. Part 2 makes it trustworthy. This lesson is about **context engineering**: what the model knows when it decides, and what it should do when it doesn't know enough.

## Where lesson 5 left off

Lesson 5's control test showed something worrying. With memory switched off, the agent never asked for information it didn't have. It guessed:

- "Log 3 hours on my main project" was saved to **Cornerpost Watch**, the first name on its list.
- "Spent 2 hours on the company site" was saved to **Agentic Engineering Worklog**, with no warning.

A guess gets saved as if it were fact. Memory fixed those two cases, but only because the earlier message happened to be in the conversation. The agent needs a better habit: **when it doesn't know, ask.**

## The big idea: prompt engineering vs. context engineering

These two terms come up constantly, and this lesson uses both:

- **Prompt engineering** is about the *instructions*: how you tell the model what to do. "How should I phrase this?"
- **Context engineering** is about the *information*: everything the model can see when it decides, which is the instructions, the tool descriptions, the conversation so far and any results from tools. "What does the model need to know right now?"

The model only knows what's in its context (lesson 1). When the context doesn't hold the answer, the model has two choices: guess or ask. Left alone, lesson 5 showed it guesses. This lesson tries two fixes, one after the other, and measures each:

1. **An instruction** (prompt engineering): the **ask rule**, telling the model to ask when the project isn't clear.
2. **A check in code**: the **named-project check**, which refuses to save an entry unless you named its project somewhere in the conversation.

The first run of this lesson, on November 22, 2026, gave a clear answer about the first fix: the model ignored the instruction completely. That result is shown below, and it's the reason for the second fix.

## What we are looking for

By the end of this notebook you should be able to say yes to five things:

1. I can see exactly what the model is sent on each call: its context.
2. I know what the ask rule says, and I've seen whether the model follows it.
3. I know the difference between *asking* the model to do something and *making sure* it happens in code.
4. With the check in code, no entry is saved under a project I never named.
5. I can read a test with controls and say which change made the difference.

**How to run it in PyCharm:** use **Run All** (the double-arrow button at the top of the notebook), wait until the last cell finishes, then save with **Ctrl + S**. Steps 5 and 8 make about 80 model calls between them, so allow 10 minutes or more.

## Step 1: Setup

### What we are looking for

The notebook using the `worklog` package with a demo log, as in lesson 5, so nothing touches your real work log.

In [1]:
import re
import sys
from datetime import date, timedelta
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from worklog import config, tools
from worklog.agent import ASK_RULE, NOT_NAMED, Conversation, build_system_prompt, project_was_named

DEMO_LOG = config.DATA_DIR / "demo_log_lesson6.jsonl"
tools.set_log_file(DEMO_LOG)

TODAY_S = date.today().isoformat()
YESTERDAY = (date.today() - timedelta(days=1)).isoformat()

STARTING_ENTRIES = [
    {"logged_at": f"{YESTERDAY}T09:00:00", "project": "Cornerpost Watch", "description": "Tuned the cloud mask", "hours": 1},
    {"logged_at": f"{YESTERDAY}T14:00:00", "project": "Agentic Engineering Worklog", "description": "Ran lesson 5", "hours": None},
    {"logged_at": f"{TODAY_S}T08:30:00", "project": "Cornerpost Watch", "description": "Fixed the alert email bug", "hours": 2},
    {"logged_at": f"{TODAY_S}T10:00:00", "project": "Herd Planner", "description": "Added vaccine records", "hours": 1.5},
]


def reset_log():
    """Put the demo log back to the four starting entries."""
    tools.write_entries(STARTING_ENTRIES)


reset_log()
print("Model:            ", config.MODEL)
print("Using:            ", tools.get_log_file())
print("Entries:          ", len(tools.read_entries()))
print("Ask when unclear: ", config.ASK_WHEN_UNCLEAR)
print("Require named project:", config.REQUIRE_NAMED_PROJECT)

Model:             qwen3:8b
Using:             B:\Development\Python\Agentic-Engineering-Worklog\data\demo_log_lesson6.jsonl
Entries:           4
Ask when unclear:  True
Require named project: True


### The code, line by line

- The first part finds the project folder and adds it to `sys.path`, so `import worklog` works from the `notebooks` folder (lesson 5, Step 1).
- `from worklog.agent import ...` loads several things from the package:
    - `ASK_RULE` is the new instruction this lesson adds.
    - `Conversation` is the agent with memory (lessons 4 and 5).
    - `build_system_prompt` builds the full instructions, so we can look at them.
    - `NOT_NAMED` and `project_was_named` are the check in code that Step 7 adds.
- `tools.set_log_file(DEMO_LOG)` points every tool at this lesson's demo log.
- `STARTING_ENTRIES` and `reset_log()` give every test the same known starting point, as in lessons 4 and 5.
- The last two lines print the two new settings, `config.ASK_WHEN_UNCLEAR` (the instruction) and `config.REQUIRE_NAMED_PROJECT` (the check in code).

### How to read it

You should see the model `qwen3:8b`, a log ending in `demo_log_lesson6.jsonl`, **4** entries, **Ask when unclear: True** and **Require named project: True**.

### What it means

Both fixes are now part of the package, switched on by default in `config.py`, so the terminal program, `python -m worklog`, uses them from now on. The tests below switch them off on purpose, as controls.

### What's next and why

Before changing what the model does, Step 2 looks at exactly what it sees.

## Step 2: What the model actually sees

### What we are looking for

The complete list of messages sent to the model on a call, its **context**, and how big it is.

In [2]:
reset_log()
chat = Conversation()
chat.send("Everything I log next is for Cornerpost Logic.")

next_message = {"role": "user", "content": "Spent 2 hours on the company site."}
context = [{"role": "system", "content": chat.system_prompt}] + chat.recent_messages() + [next_message]

print(f"The model would be sent {len(context)} messages:")
print()
for number, message in enumerate(context, start=1):
    role = message["role"] if isinstance(message, dict) else message.role
    content = message["content"] if isinstance(message, dict) else (message.content or "")
    shown = content.replace("\n", " ")
    print(f"{number}. {role:<9} {len(content):>5} characters   {shown[:90]}{'...' if len(shown) > 90 else ''}")
print()
print("Tools described to the model:", ", ".join(t["function"]["name"] for t in tools.TOOLS))
print("Tokens the model read on the first message:", chat.prompt_tokens[0])

The model would be sent 4 messages:

1. system      873 characters   You are a work log assistant. Today is 2026-11-23 (Monday). When the user describes work t...
2. user         46 characters   Everything I log next is for Cornerpost Logic.
3. assistant   104 characters   Got it! All your future logs will be for Cornerpost Logic. Let me know when you'd like to ...
4. user         34 characters   Spent 2 hours on the company site.

Tools described to the model: add_entry, search_entries, update_last_entry
Tokens the model read on the first message: 627


### The code, line by line

- `chat = Conversation()` creates an agent with the default settings: memory of 5 turns, and the ask rule on.
- `chat.send(...)` sends the first message, the one that names the project.
- `context` rebuilds exactly what the agent would send for the *next* message. It's the same three parts as the agent's own `send` method:
    1. the **system** message: the standing instructions, `chat.system_prompt`
    2. the **recent turns** the agent remembers, `chat.recent_messages()`
    3. the **new user message**
- The loop prints one line per message: its number, who it's from (`role`), its length in characters, and the start of its text. Some messages are dictionaries and some are the model's own reply objects, which is why the code checks `isinstance`.
- The last two lines list the tools the model is told about, and how many **tokens** (pieces of words, lesson 1) the model read for the first message.

### How to read it

You should see about **4 messages**:

1. **system**: by far the longest, at about 900 characters. These are the instructions: today's date, the rules, the known projects and the new ask rule.
2. **user**: "Everything I log next is for Cornerpost Logic."
3. **assistant**: the agent's reply to it.
4. **user**: "Spent 2 hours on the company site."

The tool descriptions are sent too, as a separate part of the call, which is why the token count is much larger than the messages alone suggest.

### What it means

This list **is** everything the model knows. It has no other memory and no access to anything else. "The company site" can only be understood because message 2 is in the list. Take message 2 away, which is what switching memory off does, and the model has no way to know.

That's context engineering in one picture: decide what goes into this list. Too little, and the model guesses. Too much, and every call is slower and costlier (lesson 4 measured this: +62% by turn 5).

### What's next and why

When the list doesn't hold the answer, the model needs an instruction for what to do. Step 3 looks at that instruction.

## Step 3: The ask rule

### What we are looking for

The exact words of the new instruction, and where it sits among the rest.

In [3]:
print("The ask rule:")
print(" ", ASK_RULE)
print()

with_rule = build_system_prompt(ask_when_unclear=True)
without_rule = build_system_prompt(ask_when_unclear=False)
print("Instructions with the rule:   ", len(with_rule), "characters")
print("Instructions without the rule:", len(without_rule), "characters")
print("Difference:                   ", len(with_rule) - len(without_rule), "characters")
print()
start = with_rule.index(ASK_RULE)
print("It comes right after the list of projects:")
print("  ..." + with_rule[start - 80:start + 60] + "...")

The ask rule:
  If it isn't clear which known project the work was for, from the message or the conversation so far, do not guess and do not call add_entry: ask the user which project it was, then wait for the answer. If the project is clear, don't ask. 

Instructions with the rule:    873 characters
Instructions without the rule: 635 characters
Difference:                    238 characters

It comes right after the list of projects:
  ... Agentic Engineering Worklog. Use one of those names when the work matches one. If it isn't clear which known project the work was for, from...


### The code, line by line

- `ASK_RULE` is a piece of text defined in `worklog\agent.py`. Open that file to see it next to the other instructions.
- `build_system_prompt(ask_when_unclear=True)` builds the instructions with the rule; `False` builds them without it. This switch is what lets the test in Step 5 use a control.
- `len(...)` counts characters, to show how much the rule adds.
- The last lines find where the rule sits and print the text around it.

### How to read it

The rule says, in short: *if it isn't clear which known project the work was for, don't guess and don't save; ask, and wait. If it is clear, don't ask.* It adds about 240 characters and sits right after the list of known projects, the information it refers to.

### What it means

The rule has two halves, and both matter:

- **"Ask when unclear"** prevents wrong entries.
- **"Don't ask when it's clear"** prevents an agent that interrupts all the time. An agent that asks about everything is as useless as one that guesses about everything.

The test in Step 5 checks both halves.

Notice too what the rule does **not** do: nothing in code stops a wrong save. The rule is a request in the instructions, and the model may or may not follow it. Lesson 7 adds checks in code, called **guardrails**, that are enforced whatever the model decides.

### What's next and why

Step 4 tries the rule on lesson 5's worst case.

## Step 4: Try it

### What we are looking for

Whether the instruction alone makes the agent ask which project "the company site" means. The check in code is switched off here, so we see the instruction on its own.

In [4]:
reset_log()
chat = Conversation(verbose=True, require_named_project=False)

for text in ["Spent 2 hours on the company site.", "Cornerpost Logic."]:
    print("You:  ", text)
    reply, calls = chat.send(text)
    print("Agent:", reply.strip())
    print("       new entries so far:", len(tools.read_entries()) - len(STARTING_ENTRIES))
    print()

print("Last entry:", tools.read_entries()[-1])

You:   Spent 2 hours on the company site.
  [step 1] add_entry({'description': 'Worked on the company site', 'hours': 2, 'project': 'Agentic Engineering Worklog'}) -> Saved entry for Agentic Engineering Worklog.
Agent: The entry for the Agentic Engineering Worklog has been saved. Let me know if you need anything else!
       new entries so far: 1

You:   Cornerpost Logic.
Agent: Could you please clarify what you'd like to log or what you need assistance with?
       new entries so far: 1

Last entry: {'logged_at': '2026-11-23T17:42:06', 'project': 'Agentic Engineering Worklog', 'description': 'Worked on the company site', 'hours': 2}


### The code, line by line

- `Conversation(verbose=True, require_named_project=False)` creates an agent that prints each tool step, with the ask rule on and the check in code **off**.
- The loop sends two messages: an unclear one, then a one-word answer.
- After each reply it prints how many entries have been added to the log since the start.
- The last line prints the newest entry.

### How to read it

If the rule worked, the first message would get a question and **0** new entries, and the answer "Cornerpost Logic." would then save 2 hours to Cornerpost Logic.

In the first run it didn't work:

1. "Spent 2 hours on the company site." → an `add_entry` step saving **2 hours to Agentic Engineering Worklog**: a guess, with **1** new entry and no question.
2. "Cornerpost Logic." → the agent had no question waiting for an answer, so it replied "Could you please clarify what you'd like to log or ask about?" and nothing changed.

The last entry is the wrong one: Agentic Engineering Worklog, "Worked on the company site", 2 hours.

### What it means

The instruction was in the context (Step 3 showed exactly where), and the model still guessed. A model doesn't *obey* instructions the way code runs; it writes the reply its training makes likely, and instructions only shift those odds. For this small model, on this message, the shift wasn't enough.

### What's next and why

One example could be bad luck. Step 5 tests the rule on nine conversations, with controls.

## Step 5: A test with two controls

### What we are looking for

Nine test conversations, each run three ways:

| Setting | Ask rule | Memory | What it shows |
|---|---|---|---|
| **A. Before** | off | on | How the agent behaved before this lesson: the first control |
| **B. With the rule** | on | on | The agent as it is now |
| **C. Rule, no memory** | on | off | Whether it asks instead of guessing when the context really is missing: the second control |

Comparing A with B shows what the rule changed. Comparing B with C shows what memory still adds.

In [5]:
TEST_CONVERSATIONS = [
    {"name": "Clear: Herd Planner named",
     "messages": ["Spent 2 hours on Herd Planner fixing the calving chart."],
     "expect": ("save", [("Herd Planner", 2)])},
    {"name": "Clear: Cornerpost Watch named",
     "messages": ["Fixed the alert email bug in Cornerpost Watch, about 2 hours."],
     "expect": ("save", [("Cornerpost Watch", 2)])},
    {"name": "Not about work",
     "messages": ["What is the capital of Colorado?"],
     "expect": ("answer", r"denver")},
    {"name": "Unclear: the company site",
     "messages": ["Spent 2 hours on the company site."],
     "expect": ("ask", None)},
    {"name": "Unclear: my main project",
     "messages": ["Log 3 hours on my main project."],
     "expect": ("ask", None)},
    {"name": "Unclear: no project at all",
     "messages": ["Worked for an hour this morning."],
     "expect": ("ask", None)},
    {"name": "Made clear in an earlier message",
     "messages": ["Everything I log next is for Cornerpost Logic.", "Spent 2 hours on the company site."],
     "expect": ("save", [("Cornerpost Logic", 2)]),
     "expect_without_memory": ("ask", None)},
    {"name": "Main project named earlier",
     "messages": ["My main project this week is Herd Planner.", "Log 3 hours on my main project."],
     "expect": ("save", [("Herd Planner", 3)]),
     "expect_without_memory": ("ask", None)},
    {"name": "Answering its question",
     "messages": ["Spent 2 hours on the company site.", "Cornerpost Logic."],
     "expect": ("save", [("Cornerpost Logic", 2)]),
     "expect_without_memory": ("ask", None)},
]

SETTINGS = [
    ("A. Before", False, 5),
    ("B. With the rule", True, 5),
    ("C. Rule, no memory", True, 0),
]


def same_hours(got, expected):
    try:
        return float(got) == float(expected)
    except (TypeError, ValueError):
        return False


def run_case(case, ask_when_unclear, max_turns, require_named_project=False):
    """Run one conversation. Returns what happened and whether it matched what was expected."""
    reset_log()
    chat = Conversation(max_turns=max_turns, ask_when_unclear=ask_when_unclear,
                        require_named_project=require_named_project)
    reply = ""
    for text in case["messages"]:
        reply, _ = chat.send(text)
    reply = (reply or "").strip()
    entries = tools.read_entries()
    start_ok = entries[:len(STARTING_ENTRIES)] == STARTING_ENTRIES
    new = [(e["project"], e["hours"]) for e in entries[len(STARTING_ENTRIES):]]

    if new:
        outcome = "saved " + ", ".join(f"{p} {h}" for p, h in new)
    elif "?" in reply:
        outcome = "asked"
    else:
        outcome = "answered"

    kind, detail = case["expect"] if max_turns > 0 else case.get("expect_without_memory", case["expect"])
    if kind == "save":
        passed = len(new) == len(detail) and all(g[0] == w[0] and same_hours(g[1], w[1]) for g, w in zip(new, detail))
    elif kind == "ask":
        passed = not new and "?" in reply
    else:
        passed = not new and re.search(detail, reply, re.IGNORECASE) is not None
    return {"passed": passed and start_ok, "outcome": outcome, "expected": kind, "reply": reply,
            "first_tokens": chat.prompt_tokens[0], "blocked": chat.blocked_calls}


results = {}
for setting, ask, turns in SETTINGS:
    for case in TEST_CONVERSATIONS:
        results[(setting, case["name"])] = run_case(case, ask, turns)

print(f"{'Test':<34} | {'A. Before':<34} | {'B. With the rule':<34} | {'C. Rule, no memory':<34}")
print("-" * 145)
for case in TEST_CONVERSATIONS:
    cells = []
    for setting, _, _ in SETTINGS:
        r = results[(setting, case["name"])]
        cells.append(f"{'pass' if r['passed'] else 'FAIL'}  {r['outcome']}"[:34])
    print(f"{case['name']:<34} | {cells[0]:<34} | {cells[1]:<34} | {cells[2]:<34}")
print()

for setting, _, _ in SETTINGS:
    runs = [results[(setting, c["name"])] for c in TEST_CONVERSATIONS]
    passed = sum(r["passed"] for r in runs)
    wrong_saves = sum(1 for r in runs if r["outcome"].startswith("saved") and not r["passed"])
    needless_questions = sum(1 for r in runs if r["outcome"] == "asked" and r["expected"] != "ask")
    print(f"{setting:<20} passed {passed} of {len(runs)}   wrong saves: {wrong_saves}   needless questions: {needless_questions}")
reset_log()

Test                               | A. Before                          | B. With the rule                   | C. Rule, no memory                
-------------------------------------------------------------------------------------------------------------------------------------------------
Clear: Herd Planner named          | pass  saved Herd Planner 2         | pass  saved Herd Planner 2         | pass  saved Herd Planner 2        
Clear: Cornerpost Watch named      | pass  saved Cornerpost Watch 2     | pass  saved Cornerpost Watch 2     | pass  saved Cornerpost Watch 2    
Not about work                     | pass  answered                     | pass  answered                     | pass  answered                    
Unclear: the company site          | FAIL  saved Agentic Engineering Wo | FAIL  saved Agentic Engineering Wo | FAIL  saved Agentic Engineering Wo
Unclear: my main project           | FAIL  saved Cornerpost Watch 3     | FAIL  saved Cornerpost Watch 3     | FAIL  saved C

### The code, line by line

- `TEST_CONVERSATIONS` has nine conversations in three groups:
    - **Clear (3):** the project is named, or the message isn't about work. The right behaviour is to save, or just answer, **without asking**.
    - **Unclear (3):** the message doesn't say which project. The right behaviour is to **ask**, and save nothing.
    - **Made clear earlier (3):** the project is in an earlier message, or in the answer to the agent's own question. With memory the right behaviour is to **save**; without memory the information is gone, so the right behaviour is to **ask** (`expect_without_memory`).
- `SETTINGS` lists the three ways each conversation is run: the rule on or off, and memory 5 turns or 0.
- `run_case` runs one conversation from the known starting log, with the check in code **off** unless asked for (Step 8 switches it on), then decides what happened:
    - **saved** (and what), if any new entries were added
    - **asked**, if nothing was saved and the reply contains a question mark
    - **answered**, otherwise
- It then checks that against what was expected:
    - a **save** passes only if exactly the right project and hours were saved
    - an **ask** passes only if nothing was saved and the reply asks a question
    - an **answer** passes if nothing was saved and the reply contains the right fact
- The table prints one row per conversation, with pass or FAIL and what happened under each setting.
- The last lines count three things for each setting:
    - **passed:** how many of the nine went right
    - **wrong saves:** an entry was saved when it shouldn't have been, or with the wrong project. This is the harmful mistake.
    - **needless questions:** it asked when it should have just acted. This is the annoying mistake.

### How to read it

Look first at the **wrong saves** count. That's the number this lesson is meant to bring to zero. Then look for the word **asked** anywhere in the table.

The first run's totals were:

| Setting | Passed | Wrong saves | Questions asked |
|---|---|---|---|
| **A. Before** | 4 of 9 | 5 | 0 |
| **B. With the rule** | 5 of 9 | 4 | 0 |
| **C. Rule, no memory** | 3 of 9 | 6 | 0 |

The agent **never asked a question**, in any of the 27 runs. All three unclear messages were saved under a guess in every setting: "the company site" and "no project at all" under Agentic Engineering Worklog, and "my main project" under Cornerpost Watch, the first name on the list (lesson 5's finding again).

The one difference between A and B, "Made clear in an earlier message", is probably not the rule's doing. Under A the model saved Cornerpost Logic with **no hours** and a second entry; under B it saved the right one. A single difference like that can come from any small change in the context.

To see a reply in full, print it, for example `results[("B. With the rule", "Unclear: my main project")]["reply"]`.

### What it means

The two controls do different jobs:

- **A vs. B** isolates the rule. Same memory, rule off then on, so any difference is the rule's doing. The difference was one conversation, and the agent still never asked. **The instruction didn't work.**
- **B vs. C** isolates memory. Same rule, memory on then off. Switching memory off added two more wrong saves, just as in lesson 5: without the earlier message, the agent guessed instead of asking.

This is a real finding, and an important one in agentic engineering: **an instruction is a request, not a guarantee.** Writing a rule into the prompt and seeing it work in one hand-picked example (as the stand-in model did while this lesson was being built) proves nothing. Only a test with a control shows whether a rule changes behaviour.

### What's next and why

Before fixing it, Step 6 measures what the rule costs, because an instruction that doesn't work still costs tokens on every call.

## Step 6: What the rule costs

### What we are looking for

How many extra tokens the rule adds to every call, measured on the same message with the rule off and on.

In [6]:
case = "Clear: Herd Planner named"
before = results[("A. Before", case)]["first_tokens"]
after = results[("B. With the rule", case)]["first_tokens"]
print(f"Tokens read for '{TEST_CONVERSATIONS[0]['messages'][0]}'")
print(f"  without the rule: {before}")
print(f"  with the rule:    {after}")
print(f"  difference:       {after - before} tokens ({(after - before) / before:+.0%})")

Tokens read for 'Spent 2 hours on Herd Planner fixing the calving chart.'
  without the rule: 576
  with the rule:    632
  difference:       56 tokens (+10%)


### The code, line by line

- Step 5 saved how many tokens the model read on the first call of every run, in `first_tokens`.
- This cell compares the same message, "Spent 2 hours on Herd Planner fixing the calving chart.", under setting A (rule off) and setting B (rule on). Everything else in the two calls is identical, so the difference is the rule alone.
- `:+.0%` prints the difference as a percentage with a plus sign.

### How to read it

In the first run: **576** tokens without the rule, **632** with it, a difference of **56 tokens (+10%)**.

### What it means

Every word in the instructions is read again on every call, for every message, forever. So each instruction should earn its place. This one cost 10% more on every call and changed almost nothing, so on its own it doesn't earn its place.

It stays in the package anyway, because it describes the behaviour we want, and a larger model may follow it. But the protection has to come from somewhere the model can't ignore.

### What's next and why

Step 7 moves the rule out of the instructions and into code.

## Step 7: Enforce it in code

### What we are looking for

A check that runs **before** a tool does, in Python, so the model can't ignore it: a new entry is only saved if you named its project somewhere in the conversation. Then lesson 5's worst case, tried again with the check on.

In [7]:
import inspect

print(inspect.getsource(project_was_named))
print(inspect.getsource(Conversation.check_call))
print("What the model is told when a save is refused:")
print(" ", NOT_NAMED)
print()

reset_log()
chat = Conversation(verbose=True)   # the check is on by default

for text in ["Spent 2 hours on the company site.", "Cornerpost Logic."]:
    print("You:  ", text)
    reply, calls = chat.send(text)
    print("Agent:", reply.strip())
    print("       new entries so far:", len(tools.read_entries()) - len(STARTING_ENTRIES))
    print()

print("Saves refused by the check:", chat.blocked_calls)
print("Last entry:", tools.read_entries()[-1])

def project_was_named(project, user_texts):
    """True if the project's name appears in anything the user said (lesson 6).

    This is a check in code, not an instruction: the model can't talk its way past it.
    It only checks that the name was said, not that it was said about this work.
    """
    said = " ".join(user_texts).lower()
    return bool(project) and project.lower() in said

    def check_call(self, name, arguments, turn):
        """Checks in code before a tool runs. Returns a refusal message, or None to go ahead."""
        if self.require_named_project and name == "add_entry":
            said = [m["content"] for m in self.recent_messages() + turn
                    if isinstance(m, dict) and m.get("role") == "user"]
            if not project_was_named(arguments.get("project"), said):
                return NOT_NAMED
        return None

What the model is told when a save is refused:
  Not saved: the user hasn't said which project this work was for. Ask the user w

### The code, line by line

- `inspect.getsource(...)` prints a function's own code, so you can read the check here instead of opening `worklog\\agent.py`.
- **`project_was_named(project, user_texts)`** joins everything the user said, makes it lower case, and checks whether the project's name appears in it. It returns `True` or `False`.
- **`Conversation.check_call(...)`** runs before every tool. If the check is on and the tool is `add_entry`, it collects the user's messages from the remembered turns and the current one, and asks `project_was_named`. If the name was never said, it returns a refusal message instead of letting the tool run.
- In the agent's loop (`send`), a refusal is sent back to the model as the tool's result, in place of "Saved entry for ...", and `blocked_calls` counts it. The log file is never touched.
- `NOT_NAMED` is that refusal message. It tells the model *why* nothing was saved and what to do instead: ask.
- The last part repeats Step 4's conversation, this time with the check on.

### How to read it

What should happen:

1. "Spent 2 hours on the company site." → the model still guesses and asks for `add_entry` with a project you never named. The step line shows the result **"Not saved: the user hasn't said which project..."**, and **0** new entries. With that result in front of it, the model should then ask which project.
2. "Cornerpost Logic." → the model asks for `add_entry` again, now for Cornerpost Logic. You've just said that name, so the check lets it through: **1** new entry, Cornerpost Logic, **2** hours.

**Saves refused by the check** should be at least 1.

### What it means

Notice what changed and what didn't. The model is the same, and it still guesses. But the guess can no longer reach the log: the code stands between the model's request and the file, the same way lesson 2's lookup table decides which functions can run at all.

This is the difference between the two fixes:

| | The ask rule (Step 3) | The named-project check (this step) |
|---|---|---|
| Where it lives | In the instructions | In Python, before the tool runs |
| Who enforces it | The model, if it chooses to | The code, every time |
| What it costs | About 56 tokens on every call | Nothing, unless a save is refused; then one extra model call |
| Can the model get around it? | Yes: it ignored it in the first run | No |

The check also feeds the refusal back as context, which is context engineering again: the model gets the one piece of information it was missing, "you don't know the project", at the moment it needs it.

**Its limit:** it only checks that the name was *said*, not that it was said *about this work*. "I didn't touch Herd Planner today; spent 2 hours on the company site" would let a Herd Planner guess through. Lesson 7 adds more checks; no single check catches everything.

### What's next and why

One example again. Step 8 runs the full test with the check on, as a third control.

## Step 8: Test again with the check

### What we are looking for

The same nine conversations with the check on, with and without memory, beside Step 5's setting B (the rule alone):

| Setting | Ask rule | Check in code | Memory |
|---|---|---|---|
| **B. With the rule** (from Step 5) | on | off | on |
| **D. Rule and check** | on | **on** | on |
| **E. Rule and check, no memory** | on | **on** | off |

B vs. D isolates the check. D vs. E shows what memory still adds.

In [8]:
CHECK_SETTINGS = [
    ("D. Rule and check", True, 5),
    ("E. Rule and check, no memory", True, 0),
]
for setting, ask, turns in CHECK_SETTINGS:
    for case in TEST_CONVERSATIONS:
        results[(setting, case["name"])] = run_case(case, ask, turns, require_named_project=True)

COMPARE = ["B. With the rule", "D. Rule and check", "E. Rule and check, no memory"]
print(f"{'Test':<34} | {'B. With the rule':<34} | {'D. Rule and check':<34} | {'E. Rule and check, no memory':<34}")
print("-" * 145)
for case in TEST_CONVERSATIONS:
    cells = []
    for setting in COMPARE:
        r = results[(setting, case["name"])]
        cells.append(f"{'pass' if r['passed'] else 'FAIL'}  {r['outcome']}"[:34])
    print(f"{case['name']:<34} | {cells[0]:<34} | {cells[1]:<34} | {cells[2]:<34}")
print()

for setting in COMPARE:
    runs = [results[(setting, c["name"])] for c in TEST_CONVERSATIONS]
    passed = sum(r["passed"] for r in runs)
    wrong_saves = sum(1 for r in runs if r["outcome"].startswith("saved") and not r["passed"])
    asked = sum(1 for r in runs if r["outcome"] == "asked")
    blocked = sum(r["blocked"] for r in runs)
    print(f"{setting:<30} passed {passed} of {len(runs)}   wrong saves: {wrong_saves}   asked: {asked}   saves refused: {blocked}")
reset_log()

Test                               | B. With the rule                   | D. Rule and check                  | E. Rule and check, no memory      
-------------------------------------------------------------------------------------------------------------------------------------------------
Clear: Herd Planner named          | pass  saved Herd Planner 2         | pass  saved Herd Planner 2         | pass  saved Herd Planner 2        
Clear: Cornerpost Watch named      | pass  saved Cornerpost Watch 2     | pass  saved Cornerpost Watch 2     | pass  saved Cornerpost Watch 2    
Not about work                     | pass  answered                     | pass  answered                     | pass  answered                    
Unclear: the company site          | FAIL  saved Agentic Engineering Wo | pass  asked                        | pass  asked                       
Unclear: my main project           | FAIL  saved Cornerpost Watch 3     | pass  asked                        | pass  asked  

### The code, line by line

- `CHECK_SETTINGS` lists the two new settings. Both have the rule on; the difference is memory.
- The loop runs every conversation under each new setting with `require_named_project=True`, and adds the results to the same `results` dictionary as Step 5, so B, D and E can be printed side by side.
- The table and totals work like Step 5's, with two new counts:
    - **asked:** how many conversations ended with the agent asking a question
    - **saves refused:** how many times the check stopped a save

### How to read it

What should happen:

- **Wrong saves** drop to **0** under D and E, or very close. A wrong save is now only possible when the guessed project was named somewhere in the conversation.
- The three **unclear** conversations change from "saved ..." to **asked** (or, if the model keeps retrying instead of asking, to "answered" with nothing saved).
- The three **clear** conversations still save, with **0** refusals: the check doesn't get in the way when you named the project.
- Under E, the last three should ask, because without memory the project you named earlier is gone.

### What it means

Compare B with D: the same model and the same instructions, plus about a dozen lines of Python. Whatever that changes, the check did it. If wrong saves fall to zero, the check made it impossible to file work under a project you never named, something the instruction alone couldn't do.

That's the main lesson of Part 2 so far: **put the rules that matter in code.** Use instructions to describe what you'd like; use checks to make sure of what you need.

### What's next and why

That's the end of the lesson's steps. Below are exercises, a summary, and what comes next.

## Exercises

1. **Use it for real.** Run `python -m worklog` and log some work without naming the project. Does it ask? Is the question clear?
2. **Find the check's limit.** Write a message that gets a wrong project past the named-project check (Step 7 gives a hint). Add it to `TEST_CONVERSATIONS`.
3. **Too many questions?** Write a message where the project is clear but described indirectly ("the cattle app"). Does it ask needlessly? Should it?
4. **Change the wording.** Edit `ASK_RULE` in `worklog\agent.py` to be firmer or shorter, restart the notebook, and rerun Step 5. Can any wording make this model ask?

## What it all means

- **The context is everything the model knows:** instructions, tool descriptions, the recent conversation and tool results. Context engineering is deciding what goes in it.
- **When the context lacks the answer, the model must guess or ask.** Without instructions, it guesses and saves the guess.
- **An instruction is a request, not a guarantee.** The ask rule was in the context, and the model never asked once in 27 runs. Only a test with a control showed that.
- **Instructions cost tokens on every call** (+10% here), whether they work or not.
- **A check in code can't be ignored.** The named-project check stops a save unless you named the project, and tells the model to ask.
- **Controls showed which change did what:** the rule (A vs. B), memory (B vs. C) and the check (B vs. D).
- **Every check has limits.** This one only checks that a name was said. Lesson 7 adds more.

## Words used in this lesson

| Word | Meaning |
|---|---|
| Context | Everything sent to the model on one call: instructions, tool descriptions, recent conversation and tool results. |
| Context engineering | Deciding what goes into the context, so the model has what it needs and no more. |
| Prompt engineering | Deciding how the instructions are worded. |
| System message | The first message in the context: the agent's standing instructions. |
| Ask rule | This lesson's instruction: ask which project when it isn't clear, and don't ask when it is. |
| Clarifying question | A question the agent asks to fill in missing information before acting. |
| Human in the loop | Building in points where the agent hands a decision back to a person. |
| Control | The same test with one thing switched off, to see what that thing really does. |
| Wrong save | An entry saved with the wrong project, or saved when it shouldn't have been. |
| Needless question | Asking when the answer was already clear. |
| Guardrail | A check enforced in code, whatever the model decides. |
| Named-project check | This lesson's guardrail: a new entry is only saved if the user named its project in the conversation. |
| Refusal | What a check sends back to the model in place of a tool's result, saying why nothing happened and what to do instead. |

## What's next and why

**Lesson 7: Guardrails, approval and security.** This lesson added one check in code. Lesson 7 adds the rest:

- a project must be on the known list
- hours must be sensible
- changing an entry needs your "yes"

It also shows **prompt injection**: how text hidden in data can try to give the agent orders, and how guardrails stop it.